# 03 · Ouro (modelo dimensional + indicadores do MVP)

```
                dim_tempo (sk_tempo)
                       │
dim_ciclovia ──── fato_relato ──── dim_tipo_perigo
(ciclovia_id)   1 linha por relato   (tipo_perigo_id)
                + chuva daquela hora
```

- **Relato → ciclovia:** menor distância do ponto até um segmento de ciclovia; até
  `RAIO_M` metros conta como "na ciclovia", acima disso vira `-1 · Fora da rede cicloviária`.
- **Hora de chuva:** precipitação daquela hora ≥ `CHUVA_MIN_MM` (0,1 mm, a menor medida do Open-Meteo).
- **Alerta ao ciclista:** severidade `alta` + hora de chuva + dentro da rede.
- **LGPD:** `usuario_id` não entra na Ouro. No lugar dele vai
  `usuario_hash = sha2(usuario_id || salt, 256)`. O salt fica fora do repositório:
  num secret (`gs/salt_lgpd`) ou digitado no widget na hora de rodar. Usem sempre o mesmo valor.
- Relatos com status `descartado` ficam na fato, mas saem dos indicadores.

In [0]:
import math

from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("salt_lgpd", "teste")
CAT = dbutils.widgets.get("catalogo")
T = lambda nome: f"{CAT}.gs.{nome}"  # noqa: E731

RAIO_M = 30.0
CHUVA_MIN_MM = 0.1

try:
    SALT = dbutils.secrets.get("gs", "salt_lgpd")
except Exception:  # noqa: BLE001 - sem secret scope no Free Edition
    SALT = dbutils.widgets.get("salt_lgpd")
if not SALT:
    raise ValueError("Preencha o widget salt_lgpd (qualquer texto secreto; não salve no repositório).")


def gravar(df, tabela):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T(tabela))

## dim_tipo_perigo

In [0]:
dim_tipo_perigo = spark.createDataFrame(
    [
        (0, "desconhecido", "Tipo não informado", "outros"),
        (1, "buraco", "Buraco ou asfalto afundado", "infraestrutura"),
        (2, "obstrucao", "Obstrução na ciclovia", "uso indevido"),
        (3, "conflito_veiculo", "Conflito com veículo", "trânsito"),
        (4, "iluminacao_ruim", "Iluminação ruim", "infraestrutura"),
        (5, "alagamento", "Alagamento", "clima"),
        (6, "piso_escorregadio", "Piso escorregadio", "clima"),
        (7, "sinalizacao_apagada", "Sinalização apagada", "infraestrutura"),
    ],
    "tipo_perigo_id INT, tipo_perigo STRING, descricao STRING, categoria STRING",
)
gravar(dim_tipo_perigo, "dim_tipo_perigo")

## dim_ciclovia

In [0]:
dim_ciclovia = spark.table(T("prata_ciclovias")).select(
    "ciclovia_id", "nome", "tipo", F.round(F.col("comprimento_m") / 1000, 3).alias("comprimento_km"), "centro_lat", "centro_lon"
)
fora_da_rede = spark.createDataFrame(
    [("-1", "Fora da rede cicloviária", None, None, None, None)],
    dim_ciclovia.schema,
)
gravar(dim_ciclovia.unionByName(fora_da_rede), "dim_ciclovia")

## Relato → ciclovia mais próxima

In [0]:
LAT0 = -23.55
KX = 111320.0 * math.cos(math.radians(LAT0))
KY = 110574.0

relatos = spark.table(T("prata_relatos")).withColumn("px", F.col("lon") * KX).withColumn("py", F.col("lat") * KY)
segmentos = spark.table(T("prata_ciclovia_segmentos"))

# Filtro grosso pela caixa do segmento (+ raio), depois distância exata ponto-segmento.
candidatos = relatos.select("id", "px", "py").join(
    segmentos,
    (F.col("px") >= F.least("x1", "x2") - RAIO_M)
    & (F.col("px") <= F.greatest("x1", "x2") + RAIO_M)
    & (F.col("py") >= F.least("y1", "y2") - RAIO_M)
    & (F.col("py") <= F.greatest("y1", "y2") + RAIO_M),
)
dx, dy = F.col("x2") - F.col("x1"), F.col("y2") - F.col("y1")
t = ((F.col("px") - F.col("x1")) * dx + (F.col("py") - F.col("y1")) * dy) / (dx * dx + dy * dy)
t = F.when((dx * dx + dy * dy) == 0, F.lit(0.0)).otherwise(F.greatest(F.lit(0.0), F.least(F.lit(1.0), t)))
distancia = F.sqrt(
    F.pow(F.col("px") - (F.col("x1") + t * dx), 2) + F.pow(F.col("py") - (F.col("y1") + t * dy), 2)
)
mais_proxima = (
    candidatos.withColumn("distancia_m", distancia)
    .filter(F.col("distancia_m") <= RAIO_M)
    .withColumn("_n", F.row_number().over(Window.partitionBy("id").orderBy("distancia_m", "ciclovia_id")))
    .filter("_n = 1")
    .select("id", "ciclovia_id", F.round("distancia_m", 1).alias("distancia_ciclovia_m"))
)

## fato_relato

In [0]:
clima_hora = spark.table(T("prata_clima")).select(F.col("hora").alias("hora_cheia"), "precipitacao_mm")
sk = lambda col: F.date_format(col, "yyyyMMddHH").cast("bigint")  # noqa: E731

fato = (
    relatos.withColumn("hora_cheia", F.date_trunc("hour", "criado_em"))
    .join(mais_proxima, "id", "left")
    .join(clima_hora, "hora_cheia", "left")
    .join(dim_tipo_perigo.select("tipo_perigo", "tipo_perigo_id"), "tipo_perigo", "left")
    .select(
        F.col("id").alias("relato_id"),
        sk("hora_cheia").alias("sk_tempo"),
        F.coalesce("ciclovia_id", F.lit("-1")).alias("ciclovia_id"),
        F.coalesce("tipo_perigo_id", F.lit(0)).alias("tipo_perigo_id"),
        "severidade",
        "status",
        "lat",
        "lon",
        "distancia_ciclovia_m",
        "precipitacao_mm",
        (F.coalesce("precipitacao_mm", F.lit(0.0)) >= CHUVA_MIN_MM).alias("em_chuva"),
        "criado_em",
        "atualizado_em",
        F.sha2(F.concat_ws("|", "usuario_id", F.lit(SALT)), 256).alias("usuario_hash"),  # LGPD
    )
    .withColumn(
        "alerta_ciclista",
        (F.col("severidade") == "alta") & F.col("em_chuva") & (F.col("ciclovia_id") != "-1"),
    )
)
gravar(fato, "fato_relato")

## dim_tempo

In [0]:
horas = (
    spark.table(T("prata_clima")).select(F.col("hora").alias("hora_cheia"), "precipitacao_mm")
    .unionByName(
        spark.table(T("fato_relato")).select(
            F.to_timestamp(F.col("sk_tempo").cast("string"), "yyyyMMddHH").alias("hora_cheia"),
            F.lit(None).cast("double").alias("precipitacao_mm"),
        )
    )
    .groupBy("hora_cheia")
    .agg(F.max("precipitacao_mm").alias("precipitacao_mm"))
)
nomes_dia = F.element_at(
    F.array(*[F.lit(d) for d in ["domingo", "segunda", "terça", "quarta", "quinta", "sexta", "sábado"]]),
    F.dayofweek("hora_cheia"),
)
dim_tempo = horas.select(
    sk("hora_cheia").alias("sk_tempo"),
    "hora_cheia",
    F.to_date("hora_cheia").alias("data"),
    F.hour("hora_cheia").alias("hora"),
    F.dayofweek("hora_cheia").alias("dia_semana"),
    nomes_dia.alias("nome_dia"),
    F.dayofweek("hora_cheia").isin(1, 7).alias("fim_de_semana"),
    F.when(F.hour("hora_cheia") < 6, "madrugada")
    .when(F.hour("hora_cheia") < 12, "manhã")
    .when(F.hour("hora_cheia") < 18, "tarde")
    .otherwise("noite")
    .alias("periodo"),
    "precipitacao_mm",
    (F.coalesce("precipitacao_mm", F.lit(0.0)) >= CHUVA_MIN_MM).alias("hora_com_chuva"),
)
gravar(dim_tempo, "dim_tempo")

## ouro_indicadores (tabela que o MVP usa) + CSV

In [0]:
indicadores = (
    spark.table(T("fato_relato"))
    .filter(F.col("status") != "descartado")
    .join(spark.table(T("dim_ciclovia")), "ciclovia_id")
    .join(spark.table(T("dim_tipo_perigo")).select("tipo_perigo_id", F.col("descricao").alias("tipo_desc")), "tipo_perigo_id")
    .groupBy(F.col("nome").alias("ciclovia"))
    .agg(
        F.count("*").alias("total_perigos"),
        F.sum(F.when(F.col("severidade") == "alta", 1).otherwise(0)).alias("perigos_alta"),
        F.sum(F.col("em_chuva").cast("int")).alias("perigos_em_chuva"),
        F.sum(F.col("alerta_ciclista").cast("int")).alias("alertas_ciclista"),
        F.mode("tipo_desc").alias("tipo_mais_comum"),
        F.max("criado_em").alias("ultimo_relato"),
        F.round(F.avg("lat"), 6).alias("lat_media"),
        F.round(F.avg("lon"), 6).alias("lon_media"),
    )
    .withColumn("pct_em_chuva", F.round(100 * F.col("perigos_em_chuva") / F.col("total_perigos"), 1))
    .withColumn("atualizado_em", F.current_timestamp())
    .orderBy(F.desc("total_perigos"))
)
gravar(indicadores, "ouro_indicadores")

caminho_csv = f"/Volumes/{CAT}/gs/exportacao/ouro_indicadores.csv"
spark.table(T("ouro_indicadores")).orderBy(F.desc("total_perigos")).toPandas().to_csv(caminho_csv, index=False)
print(f"CSV gravado em {caminho_csv} (baixe pelo Catalog Explorer e coloque no repositório do app)")
display(spark.table(T("ouro_indicadores")))

CSV gravado em /Volumes/workspace/gs/exportacao/ouro_indicadores.csv (baixe pelo Catalog Explorer e coloque no repositório do app)


ciclovia,total_perigos,perigos_alta,perigos_em_chuva,alertas_ciclista,tipo_mais_comum,ultimo_relato,lat_media,lon_media,pct_em_chuva,atualizado_em
Fora da rede cicloviária,64,16,21,0,Buraco ou asfalto afundado,2026-11-16T18:51:31.000Z,-23.589129,-46.559123,32.8,2026-11-17T19:57:11.996Z
CICLOFAIXA CAMINHO VERDE,8,1,3,0,Buraco ou asfalto afundado,2026-11-12T06:50:55.000Z,-23.533497,-46.512643,37.5,2026-11-17T19:57:11.996Z
CICLOFAIXA ASSIS RIBEIRO,8,1,2,1,Buraco ou asfalto afundado,2026-11-15T08:01:30.000Z,-23.496155,-46.506167,25.0,2026-11-17T19:57:11.996Z
CICLOVIA RIO PINHEIROS,8,2,0,0,Buraco ou asfalto afundado,2026-11-14T08:59:21.000Z,-23.638255,-46.699772,0.0,2026-11-17T19:57:11.996Z
CICLOVIA FARIA LIMA,7,1,1,0,Alagamento,2026-11-10T17:13:07.000Z,-23.555648,-46.70812,14.3,2026-11-17T19:57:11.996Z
CICLOVIA PARQUE ECOLOGICO DO TIETE,7,3,4,1,Buraco ou asfalto afundado,2026-11-16T14:01:24.000Z,-23.482775,-46.483096,57.1,2026-11-17T19:57:11.996Z
CICLORROTA LAPA,7,3,2,1,Conflito com veículo,2026-11-15T16:34:46.000Z,-23.529482,-46.694107,28.6,2026-11-17T19:57:11.996Z
CICLOFAIXA CAETANO ALVARES,7,5,3,2,Alagamento,2026-11-16T08:34:49.000Z,-23.488532,-46.647374,42.9,2026-11-17T19:57:11.996Z
CICLOFAIXA PARQUE ARARIBA,6,3,0,0,Buraco ou asfalto afundado,2026-11-14T21:53:07.000Z,-23.629923,-46.760529,0.0,2026-11-17T19:57:11.996Z
CICLOFAIXA ARICANDUVA,6,2,2,0,Conflito com veículo,2026-11-13T18:54:43.000Z,-23.570316,-46.502282,33.3,2026-11-17T19:57:11.996Z


## Prova de LGPD (print do item 08)

In [0]:
display(spark.table(T("fato_relato")).select("relato_id", "usuario_hash", "severidade", "ciclovia_id", "em_chuva").limit(10))
print("Colunas da fato:", spark.table(T("fato_relato")).columns)

relato_id,usuario_hash,severidade,ciclovia_id,em_chuva
R20261031-008,275785efdb9845ae1562bb93f9c0ea01338325e0c4c32409950f8a98e4a324ad,media,via_bicicleta.10628,false
R20261031-009,005ee77d0db248a4c3bc290b982bcff3cd0d71a13f87333b827a8ef5ea01629b,alta,via_bicicleta.10052,false
R20261031-011,2231d8ea9df46a881574015e3deaa330692c0bae248573bde440ce23357f0510,media,via_bicicleta.11157,false
R20261031-014,1134eafab8f8accbea17459261e04fed784b963fb9cd8c872c0f763c2435f2dc,baixa,via_bicicleta.11341,false
R20261031-020,91b0dae955af47b1f15cf29a858a44abb92b69dc0414a7baa24796d8c2486519,media,via_bicicleta.10647,false
R20261031-022,b66a8124cef01f989389c902e83fd877a273eaff4528a6ef5fddb1c5b1650949,baixa,via_bicicleta.10112,false
R20261101-001,df59f03d789a0a543ade2bffc294a17dca89b5699056fd661603e13e21d9fcc2,media,via_bicicleta.11320,false
R20261101-002,306cbeb5183352ee320f8df2a19fe7e60cc85ee57c35d5200ec5c74615c27e34,baixa,via_bicicleta.11753,false
R20261101-003,eabf2a09cc22a7456ed7b02a320f21408e6cfb726a8fd4dbf56d86f081c3d4bb,baixa,via_bicicleta.11891,false
R20261101-004,5166a343f5546d2f0ab78a20e4b0a11654f1cda3f461ea447014049c60ede0f4,baixa,via_bicicleta.10334,false


Colunas da fato: ['relato_id', 'sk_tempo', 'ciclovia_id', 'tipo_perigo_id', 'severidade', 'status', 'lat', 'lon', 'distancia_ciclovia_m', 'precipitacao_mm', 'em_chuva', 'criado_em', 'atualizado_em', 'usuario_hash', 'alerta_ciclista']
